# 01 — Data Prep, Brand Selection & Intent Discovery
Self-contained notebook: no external package, edit the CONFIG cell and run top to bottom.
Set `DATA_PATH` to your local copy of the Kaggle `customer-support-on-twitter` CSV.

In [ ]:
# --- CONFIG ---
DATA_PATH = "data/raw/sample.csv"     # path to twcs.csv (or your subsample)
OUTPUT_DIR = "data/processed"
CONFIG_DIR = "config"
CANDIDATE_BRANDS = ["AmazonHelp", "AppleSupport", "SpotifyCares", "Uber_Support"]
CHOSEN_BRAND = None   # set this after reviewing the profiling table below, e.g. "AmazonHelp"
N_CLUSTERS = 10
CLUSTER_SAMPLE_SIZE = 500
RANDOM_STATE = 1


In [ ]:
import os, re, json
import pandas as pd
import numpy as np

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CONFIG_DIR, exist_ok=True)


## Step 1 — Load and profile candidate brands

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Total records: {len(df):,}")
print(f"Columns: {list(df.columns)}")

df['inbound'] = df['inbound'].astype(bool)
# consistent string IDs (response_tweet_id can hold comma-separated lists, keep as string)
df['tweet_id'] = df['tweet_id'].astype('Int64').astype(str).replace('<NA>', pd.NA)
df['in_response_to_tweet_id'] = df['in_response_to_tweet_id'].astype('Int64').astype(str).replace('<NA>', pd.NA)
df['response_tweet_id'] = df['response_tweet_id'].astype(str).replace('nan', pd.NA)

brands = df[df['inbound'] == False].copy()
customers = df[df['inbound'] == True].copy()
print(f"Brand replies: {len(brands):,} | Customer messages: {len(customers):,}")

top_brands = brands['author_id'].value_counts().head(15)
print(top_brands)


In [ ]:
boilerplate_pattern = re.compile(
    r"sorry to hear|please dm|send us a (?:dm|message)|we'd like to help|reach out via dm",
    re.IGNORECASE
)

candidates = [b for b in CANDIDATE_BRANDS if b in brands['author_id'].values] or list(top_brands.head(4).index)

records = []
for brand in candidates:
    brand_replies = brands[brands['author_id'] == brand]
    boilerplate_pct = brand_replies['text'].str.contains(boilerplate_pattern, na=False).mean() * 100
    threading_pct = brand_replies['in_response_to_tweet_id'].notna().mean() * 100
    records.append({
        "brand": brand,
        "total_replies": len(brand_replies),
        "boilerplate_pct": round(boilerplate_pct, 2),
        "threading_valid_pct": round(threading_pct, 2),
    })

profile_df = pd.DataFrame(records)
profile_df


**Decision point.** Read the table above. You want: decent volume (thousands of replies),
boilerplate ratio well under 100% (40-70% is often realistic — 95%+ leaves nothing to ground
replies on), and high threading validity (you need `in_response_to_tweet_id` to link customer
messages to replies). Set `CHOSEN_BRAND` in the CONFIG cell above and re-run from here, or just
set it directly in the next cell.

In [ ]:
CHOSEN_BRAND = CHOSEN_BRAND or candidates[0]   # override manually if needed
print("Chosen brand:", CHOSEN_BRAND)

# write the decision to the decision log as you go
decision_note = (
    f"Brand selection: chose {CHOSEN_BRAND}. "
    f"Profile: {profile_df[profile_df['brand']==CHOSEN_BRAND].to_dict('records')}"
)
with open(os.path.join(OUTPUT_DIR, "decision_notes.txt"), "a") as f:
    f.write(decision_note + "\n")
print(decision_note)


## Step 2 — Subsample to the chosen brand

In [ ]:
mask = (brands['author_id'] == CHOSEN_BRAND)
brand_tweet_ids = brands[mask]['tweet_id']
customer_tweet_ids = brands[mask]['in_response_to_tweet_id'].dropna()

subsample = df[df['tweet_id'].isin(brand_tweet_ids) | df['tweet_id'].isin(customer_tweet_ids)].copy()
print(len(subsample))
subsample.to_csv(os.path.join(OUTPUT_DIR, "raw_subsample.csv"), index=False)


## Step 3 — Reconstruct (customer_msg -> brand_reply) pairs

In [ ]:
tweet_lookup = subsample.set_index('tweet_id').to_dict('index')

pairs = []
brand_rows = subsample[(subsample['inbound'] == False) & (subsample['author_id'] == CHOSEN_BRAND)]

for _, row in brand_rows.iterrows():
    parent_id = row['in_response_to_tweet_id']
    if pd.isna(parent_id):
        continue
    parent = tweet_lookup.get(parent_id)
    if parent is None or parent['inbound'] != True:
        continue
    pairs.append({
        "customer_tweet_id": parent_id,
        "customer_text": parent['text'],
        "brand_tweet_id": row['tweet_id'],
        "brand_text": row['text'],
    })

pairs_df = pd.DataFrame(pairs)
print(len(pairs_df))
pairs_df.sample(min(5, len(pairs_df)))


## Step 4 — Clean text and tag boilerplate

In [ ]:
mention_re = re.compile(r'@\w+')
url_re = re.compile(r'https?://\S+')
whitespace_re = re.compile(r'\s+')

def clean_text(t):
    t = mention_re.sub('', str(t))
    t = url_re.sub('', t)
    t = whitespace_re.sub(' ', t).strip()
    return t

pairs_df['customer_text_clean'] = pairs_df['customer_text'].apply(clean_text)
pairs_df['brand_text_clean'] = pairs_df['brand_text'].apply(clean_text)
pairs_df['is_boilerplate'] = pairs_df['brand_text_clean'].str.contains(boilerplate_pattern, na=False)
print("Boilerplate ratio:", pairs_df['is_boilerplate'].mean())

pairs_df = pairs_df[
    (pairs_df['customer_text_clean'].str.len() > 5) &
    (pairs_df['brand_text_clean'].str.len() > 5)
]
pairs_df = pairs_df.drop_duplicates(subset=['customer_text_clean', 'brand_text_clean']).reset_index(drop=True)
print("Final cleaned pairs:", len(pairs_df))

pairs_df.to_csv(os.path.join(OUTPUT_DIR, "cleaned_pairs.csv"), index=False)


## Step 5 — Intent discovery
Cluster a sample to find natural intent categories, then hand-name them into `config/intents.yaml`.

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

cluster_sample = pairs_df.sample(n=min(CLUSTER_SAMPLE_SIZE, len(pairs_df)), random_state=RANDOM_STATE).reset_index(drop=True)
embeddings = embed_model.encode(cluster_sample['customer_text_clean'].tolist(), show_progress_bar=True)
print(embeddings.shape)

km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
cluster_sample['cluster'] = km.fit_predict(embeddings)
cluster_sample['cluster'].value_counts()


In [ ]:
for c in sorted(cluster_sample['cluster'].unique()):
    sub = cluster_sample[cluster_sample['cluster'] == c]
    print(f"=== Cluster {c} ({len(sub)} examples) ===")
    for t in sub['customer_text_clean'].sample(min(8, len(sub)), random_state=RANDOM_STATE):
        print(' -', t)
    print()


**Manual step.** Read the clusters above. Merge near-duplicates, name each surviving cluster,
then edit the `intents` dict below to reflect what you actually saw (this placeholder is just a
starting shape — do not ship it unedited).

In [ ]:
import yaml

intents = {
    "delayed_delivery": "Package or order has not arrived or is late",
    "refund_request": "Customer wants money back for an order or charge",
    "billing_dispute": "Incorrect or unexpected charge",
    "account_access": "Cannot log in or access account",
    "app_technical_issue": "App crashes, bugs, or doesn't function",
    "product_quality": "Item received damaged, wrong, or defective",
    "general_complaint": "Venting or dissatisfaction without a specific actionable issue",
    "positive_feedback": "Compliment or thanks, no action needed",
}

with open(os.path.join(CONFIG_DIR, "intents.yaml"), "w") as f:
    yaml.dump(intents, f)

print("Saved", len(intents), "intents to config/intents.yaml")


In [ ]:
# Spot-check: does every fresh random example fit somewhere reasonable?
for t in pairs_df['customer_text_clean'].sample(20, random_state=2):
    print(t)
